In [ ]:
import numpy as np
import bayesflow as bf

from bmc.helpers import build_meta_model

## Setup

The helper file `bmc/helpers.py` defines the model-comparison problem used in this notebook. It wraps three sequential-sampling simulators from `ssms` into a single meta-simulator:

- `lba3`: a three-choice Linear Ballistic Accumulator (LBA) model.
- `lca_3`: a three-choice Leaky Competing Accumulator (LCA) model with starting-point bias parameters `z0`, `z1`, and `z2`.
- `lca_no_z_3`: a three-choice LCA variant without starting-point bias.

The two LCA models ask whether the data need starting-point bias, while the LBA asks a structurally different question about whether the generating mechanism is better described by a much simpler ballistic race model rather than a leaky, noisy, competing accumulator.

The LCA side is treated as likelihood-free: we can simulate reaction times and choices, but we do not rely on an analytic likelihood for inference. The LBA is not likelihood-free in the same sense, because the likelihood exists for the LBA. Here, however, all three candidates are intentionally placed into the same BayesFlow simulation interface so the classifier sees only datasets of `(RT, choice)` pairs and learns which simulator probably generated them.

`build_meta_model(num_trials=100)` fixes each simulated dataset to 100 trials, samples parameters from the bounds in the SSMS model configs, and returns a simulator that first chooses one of the three candidate models and then generates a dataset from it.

In [3]:
meta_sim = build_meta_model(num_trials=100)

In [ ]:
train_data = meta_sim.sample(5000)
val_data = meta_sim.sample(500)

## Model Comparison Mechanics

All categorical scoring rules in `ModelComparisonWorkflow` use length-$M$ estimation heads for the $M$ candidate models. The returned evidence quantity is called `log_odds`, not because these values are log posterior odds. They equal log Bayes factors only when the model prior is uniform.

Let $\ell(\mathbf{x}) = (\ell_1(\mathbf{x}), \ldots, \ell_M(\mathbf{x}))$ denote the model logits learned for a dataset $\mathbf{x}$. Note, that statisticians do not like to call these "logits", so we side withe ML community on this terminological debate. Posterior model probabilities (PMPs) are obtained by softmax-normalizing these logits:

$$
\hat p_k(\mathbf{x}) = P(\mathcal{M}_k \mid \mathbf{x}) = \frac{\exp(\ell_k(\mathbf{x}))}{\sum_{j=1}^{M} \exp(\ell_j(\mathbf{x}))}.
$$

Log odds are then reported relative to a reference model, usually $\mathcal{M}_0$:

$$
\widehat{\log O}_{k,0}(\mathbf{x}) = \log \frac{\hat p_k(\mathbf{x})}{\hat p_0(\mathbf{x})} = \ell_k(\mathbf{x}) - \ell_0(\mathbf{x}), \qquad k = 0, \ldots, M-1,
$$

so the reference entry is zero by construction. If the model prior probabilities $\pi_k = P(\mathcal{M}_k)$ are known, posterior odds decompose into prior odds and Bayes factors:

$$
\log \frac{P(\mathcal{M}_k \mid \mathbf{x})}{P(\mathcal{M}_0 \mid \mathbf{x})} = \log \mathrm{BF}_{k,0}(\mathbf{x}) + \log \frac{\pi_k}{\pi_0}.
$$

BayesFlow therefore derives prior-adjusted log Bayes factors as

$$
\widehat{\log \mathrm{BF}}_{k,0}(\mathbf{x}) = \widehat{\log O}_{k,0}(\mathbf{x}) - \log \frac{\pi_k}{\pi_0}.
$$

With uniform priors, the prior-odds correction vanishes, so log odds and log Bayes factors coincide. With non-uniform priors, they do not. Prior model weights come from the simulator when available; otherwise BayesFlow assumes equal model priors.

In [12]:
workflow = bf.ModelComparisonWorkflow(
    simulator=meta_sim,
    summary_variables="obs",
    summary_network=bf.networks.SetTransformer(embed_dims=(16, 16), summary_dim=32),
    standardize="all"
)

## Training

In [ ]:
history = workflow.fit_offline(
    train_data,
    epochs=20,
    batch_size=32,
    validation_data=val_data,
)

## Evaluation

In [ ]:
figs = workflow.plot_default_diagnostics(test_data=val_data)

In [ ]:
table = workflow.compute_default_diagnostics(test_data=val_data)
table